# Stage 3 Qwen Baseline (Kaggle, fixed)

This notebook runs the Stage 3 baseline on Kaggle:
1. setup
2. preflight (1 sample)
3. smoke run (8 samples)
4. validate + eval smoke
5. full val_v2 run
6. validate + eval full


In [ ]:
import os
import json
import zipfile
import subprocess
from pathlib import Path

def sh(cmd: str, cwd: str | None = None, check: bool = True):
    print(f'$ {cmd}')
    p = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')
    return p

REPO_DIR = Path('/kaggle/working/vlm-for-insulator-defect-detection')
SCRIPT_RUN = REPO_DIR / 'scripts' / 'run_stage3_vlm_baseline.py'
SCRIPT_VALIDATE = REPO_DIR / 'scripts' / 'validate_vlm_labels_v1.py'
SCRIPT_EVAL = REPO_DIR / 'scripts' / 'eval_stage3_vlm_baseline.py'
CONFIG_PATH = REPO_DIR / 'configs' / 'pipeline' / 'stage3_vlm_gt_baseline.yaml'

print('Helpers ready')
print('REPO_DIR =', REPO_DIR)


In [ ]:
# 1) Clone repo if needed + install deps
if not REPO_DIR.exists():
    sh('git clone https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git /kaggle/working/vlm-for-insulator-defect-detection')
else:
    print(f'Repo already exists: {REPO_DIR}')

sh('python -m pip install -q -U transformers accelerate qwen-vl-utils')

os.chdir(REPO_DIR)
print('cwd =', os.getcwd())

# Optional environment check
sh('nvidia-smi', check=False)
sh("python -c \"import torch; print('torch', torch.__version__); print('cuda', torch.cuda.is_available())\"", check=False)


In [ ]:
# 2) Resolve VAL_JSONL from Kaggle dataset (with zip fallback)
DATASET_ROOT = Path('/kaggle/input/datasets/kostyaryazanov/idid-coco-v3')
if not DATASET_ROOT.exists():
    raise FileNotFoundError(f'Dataset root not found: {DATASET_ROOT}')

candidates = [
    DATASET_ROOT / 'stage3_regrouped_v2' / 'val' / 'vlm_labels_v1_val_v2.annotated.jsonl',
    DATASET_ROOT / 'outputs' / 'stage3_regrouped_v2' / 'val' / 'vlm_labels_v1_val_v2.annotated.jsonl',
]

VAL_JSONL = None
for c in candidates:
    if c.exists():
        VAL_JSONL = c
        break

if VAL_JSONL is None:
    found = list(DATASET_ROOT.rglob('vlm_labels_v1_val_v2.annotated.jsonl'))
    if found:
        VAL_JSONL = found[0]

if VAL_JSONL is None:
    zip_candidates = [
        DATASET_ROOT / 'stage3_regrouped_v2_kaggle_root.zip',
        DATASET_ROOT / 'stage3_regrouped_v2.zip',
    ]
    zip_path = next((z for z in zip_candidates if z.exists()), None)
    if zip_path is None:
        raise FileNotFoundError('Could not find JSONL or supported zip archive in dataset root.')
    extract_root = Path('/kaggle/working')
    print(f'Extracting: {zip_path} -> {extract_root}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_root)
    extracted = [
        extract_root / 'stage3_regrouped_v2' / 'val' / 'vlm_labels_v1_val_v2.annotated.jsonl',
        extract_root / 'outputs' / 'stage3_regrouped_v2' / 'val' / 'vlm_labels_v1_val_v2.annotated.jsonl',
    ]
    for c in extracted:
        if c.exists():
            VAL_JSONL = c
            break

if VAL_JSONL is None or not VAL_JSONL.exists():
    raise FileNotFoundError('Failed to resolve vlm_labels_v1_val_v2.annotated.jsonl')

print('VAL_JSONL =', VAL_JSONL)


In [ ]:
# 3) Preflight run (1 sample)
preflight_run_id = 'stage3_qwen_preflight_v1'
cmd = (
    f'python {SCRIPT_RUN} ' 
    f'--config {CONFIG_PATH} ' 
    '--backend-mode qwen_hf ' 
    f'--input-jsonl "{VAL_JSONL}" ' 
    f'--run-id {preflight_run_id} ' 
    '--max-samples 1 --no-resume'
)
sh(cmd, cwd=str(REPO_DIR))


In [ ]:
# 4) Smoke run (8 samples)
smoke_run_id = 'stage3_qwen_smoke_v1'
cmd = (
    f'python {SCRIPT_RUN} ' 
    f'--config {CONFIG_PATH} ' 
    '--backend-mode qwen_hf ' 
    f'--input-jsonl "{VAL_JSONL}" ' 
    f'--run-id {smoke_run_id} ' 
    '--max-samples 8 --no-resume'
)
sh(cmd, cwd=str(REPO_DIR))


In [ ]:
# 5) Validate + eval smoke
smoke_dir = REPO_DIR / 'outputs' / 'stage3_vlm_baseline_runs' / 'stage3_qwen_smoke_v1'
smoke_pred = smoke_dir / 'predictions_vlm_labels_v1.jsonl'

if smoke_pred.exists() and smoke_pred.stat().st_size > 0:
    sh(f'python {SCRIPT_VALIDATE} --input "{smoke_pred}"', cwd=str(REPO_DIR))
    sh(
        f'python {SCRIPT_EVAL} --run-dir "{smoke_dir}" --ground-truth-jsonl "{VAL_JSONL}"',
        cwd=str(REPO_DIR),
    )
else:
    print('Smoke predictions file is missing or empty:', smoke_pred)

for p in [
    smoke_dir / 'run_summary.json',
    smoke_dir / 'raw_responses.jsonl',
    smoke_dir / 'parsed_predictions.jsonl',
    smoke_dir / 'predictions_vlm_labels_v1.jsonl',
    smoke_dir / 'eval' / 'metrics.json',
    smoke_dir / 'eval' / 'review_table.csv',
]:
    print(f'{p}:', 'OK' if p.exists() else 'MISSING')


In [ ]:
# 6) Full val_v2 run
full_run_id = 'stage3_qwen_val_v2'
cmd = (
    f'python {SCRIPT_RUN} ' 
    f'--config {CONFIG_PATH} ' 
    '--backend-mode qwen_hf ' 
    f'--input-jsonl "{VAL_JSONL}" ' 
    f'--run-id {full_run_id} ' 
    '--no-resume'
)
sh(cmd, cwd=str(REPO_DIR))


In [ ]:
# 7) Validate + eval full
full_dir = REPO_DIR / 'outputs' / 'stage3_vlm_baseline_runs' / 'stage3_qwen_val_v2'
full_pred = full_dir / 'predictions_vlm_labels_v1.jsonl'

if full_pred.exists() and full_pred.stat().st_size > 0:
    sh(f'python {SCRIPT_VALIDATE} --input "{full_pred}"', cwd=str(REPO_DIR))
    sh(
        f'python {SCRIPT_EVAL} --run-dir "{full_dir}" --ground-truth-jsonl "{VAL_JSONL}"',
        cwd=str(REPO_DIR),
    )
else:
    print('Full predictions file is missing or empty:', full_pred)

for p in [
    full_dir / 'run_summary.json',
    full_dir / 'raw_responses.jsonl',
    full_dir / 'parsed_predictions.jsonl',
    full_dir / 'predictions_vlm_labels_v1.jsonl',
    full_dir / 'eval' / 'metrics.json',
    full_dir / 'eval' / 'review_table.csv',
]:
    print(f'{p}:', 'OK' if p.exists() else 'MISSING')
